# Create CIFAR Awards (multi-program fellowship pattern)

CIFAR doesn't award traditional grants — it appoints researchers to one
or more thematic Programs (e.g., "Quantum Materials", "Pan-Canadian
AI Strategy", "Brain, Mind & Consciousness") with specific roles (Fellow,
Associate Fellow, Advisor, Canada CIFAR AI Chair, etc.). We model each
appointment as one award row in the pipeline, distinguished via
`funder_scheme = program_name`. A researcher in multiple programs becomes
multiple rows. This matches the prize-pattern's (prize × laureate) idiom
but at fellowship scale.

**Awarding body:** Canadian Institute for Advanced Research — F4320309949

**Schema notes (fellowship pattern, amount waived):**
- One row per (researcher × program). 31 active programs in CIFAR's
  taxonomy; ~864 rows expected from 793 distinct researchers (most
  fellows are in 1 program, some 2-3).
- `amount` and `currency` ship as **NULL by design** — CIFAR does not
  publicly disclose per-fellow funding amounts. The §6.7 amount-coverage
  check is **WAIVED** here in the notebook header. Precedent: HHMI at
  priority 44 ("amount=NULL by design").
- `funding_type` derives from a CASE on role text (`'research'` for
  Fellow/Chair/Scholar/Investigator/Co-Director; `'other'` for
  Advisor/Advisory Committee*; `'other'` default). Script ships a
  `funding_type_hint` column with the same logic for parity; notebook
  recomputes via SQL CASE for source-of-truth.
- `lead_investigator` = the researcher (given/family from title;
  `affiliation.name` = primary institution from the WP taxonomy;
  `affiliation.country` = primary country term).
- `funder_scheme` = program display name (e.g., "Quantum Materials").
- `funder_award_id` is pre-computed as `cifar-{program_slug}-{bio_slug}`.
- `declined` boolean column populated (always False; CASE present for
  schema parity with other prize/fellowship-pattern notebooks).

**Prerequisites:**
- Run `scripts/local/cifar_to_s3.py` first to fetch from
  `cifar.ca/wp-json/wp/v2/bio` (+ taxonomies) and upload parquet to S3.

**Data source:** https://cifar.ca/wp-json/wp/v2/bio
**S3 location:** `s3a://openalex-ingest/awards/cifar/cifar_appointments.parquet`

## Step 1: Create staging table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.cifar_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/cifar/cifar_appointments.parquet`;

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.cifar_raw;

## Step 1.5: Inspect raw + §6.7 amount waiver

CIFAR does NOT publish per-fellow funding amounts. The parquet has no
`amount` column by design — `amount`/`currency` will be NULL in the
awards table, and the §6.7 amount-coverage fail-fast check is **waived**
here with the same justification as HHMI (priority 44).

Source columns from the WP REST API (100% coverage on title/program/role;
lower on institution/country since CIFAR doesn't enforce those):
`funder_award_id`, `bio_id`, `bio_slug`, `researcher_full_name`,
`researcher_given`, `researcher_family`, `program_id`, `program_name`,
`program_slug`, `role_id`, `role_name`, `all_roles`, `institution_name`,
`all_institutions`, `country_name`, `funding_type_hint`,
`landing_page_url`, `first_seen_date`, `declined`. All shipped as STRING
per runbook §1.2.5; TRY_CAST below as needed.

No runbook-style money-field discovery scan needed — there is no money
data to discover.

In [ ]:
%sql
DESCRIBE openalex.awards.cifar_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.cifar_raw LIMIT 5;

In [ ]:
%sql
-- Coverage check on key fields, split by program (sanity)
SELECT
    program_name,
    COUNT(*) AS rows,
    COUNT(researcher_full_name) AS has_name,
    COUNT(role_name) AS has_role,
    COUNT(institution_name) AS has_institution,
    COUNT(country_name) AS has_country
FROM openalex.awards.cifar_raw
GROUP BY program_name
ORDER BY rows DESC
LIMIT 15;

## Step 1.6: Fail-fast — verify CIFAR funder row exists

The transform in Step 2 does `CROSS JOIN openalex.common.funder WHERE
funder_id = 4320309949`. If that row isn't present, the CROSS JOIN
silently emits zero rows and the insert in Step 3 looks like it
succeeded. Assert presence here before transforming.

In [ ]:
%sql
-- Must return exactly 1 row with display_name containing 'Canadian Institute for Advanced Research'.
-- If 0 rows, STOP — the funder is missing from openalex.common.funder.
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320309949;

## Step 2: Transform to award schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.cifar_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320309949  -- Canadian Institute for Advanced Research
)
SELECT
    abs(xxhash64(CONCAT(
        TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
    ))) % 9000000000 AS id,
    CONCAT('CIFAR ', n.program_name, ' \u2014 ', n.researcher_full_name) AS display_name,
    CASE
        WHEN TRY_CAST(n.declined AS BOOLEAN) = TRUE AND n.role_name IS NOT NULL
            THEN CONCAT('Declined the appointment. Role: ', n.role_name)
        WHEN TRY_CAST(n.declined AS BOOLEAN) = TRUE
            THEN 'Declined the appointment.'
        WHEN n.role_name IS NOT NULL
            THEN CONCAT(n.role_name, ' in CIFAR ', n.program_name, '.')
        ELSE CONCAT('Appointment in CIFAR ', n.program_name, '.')
    END AS description,
    f.funder_id,
    n.funder_award_id,
    CAST(NULL AS DOUBLE) AS amount,
    CAST(NULL AS STRING) AS currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    -- Recompute funding_type via canonical SQL CASE (don't trust the
    -- pre-computed funding_type_hint from the source script).
    CASE
        WHEN LOWER(n.role_name) RLIKE 'advisor|advisory|committee' THEN 'other'
        WHEN LOWER(n.role_name) RLIKE 'fellow|chair|scholar|investigator|co-?director|director'
            THEN 'research'
        ELSE 'other'
    END AS funding_type,
    n.program_name AS funder_scheme,
    'cifar_wp_rest' AS provenance,
    -- CIFAR doesn't publish per-appointment start/end dates in the bio
    -- listing. Use first_seen_date (WP post creation) as a proxy for
    -- start_date when available; end_date stays NULL.
    TRY_TO_DATE(SUBSTRING(n.first_seen_date, 1, 10), 'yyyy-MM-dd') AS start_date,
    CAST(NULL AS DATE) AS end_date,
    CAST(SUBSTRING(n.first_seen_date, 1, 4) AS INT) AS start_year,
    CAST(NULL AS INT) AS end_year,
    struct(
        n.researcher_given AS given_name,
        n.researcher_family AS family_name,
        CAST(NULL AS STRING) AS orcid,
        CAST(NULL AS DATE) AS role_start,
        struct(
            n.institution_name AS name,
            n.country_name AS country,
            CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
        ) AS affiliation
    ) AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    n.landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT(
               TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
           ))) % 9000000000 AS STRING)) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM openalex.awards.cifar_raw n
CROSS JOIN funder_resolved f
WHERE n.funder_award_id IS NOT NULL
  AND n.program_name IS NOT NULL
  AND n.researcher_full_name IS NOT NULL;

## Step 3: Insert into openalex_awards_raw at priority 79

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'cifar_wp_rest' AND priority = 79;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    79 AS priority  -- CIFAR priority (matches CreateAwards.ipynb registry)
FROM openalex.awards.cifar_awards;

## Step 6: Verification

`amount`/`currency` are NULL by design — §6.7 amount-coverage check
**WAIVED** here (CIFAR doesn't publish per-fellow funding; same
justification as HHMI at priority 44). Expected: ~860 rows total,
pct_amount = 0, currency distinct count = 0. Other coverage:
100% on name/program/role; high on institution/country (most fellows
have institutional affiliations recorded).

In [ ]:
%sql
SELECT COUNT(*) AS total_cifar_award_rows FROM openalex.awards.cifar_awards;

In [ ]:
%sql
-- §6.2 Schema validation on the transformed table
DESCRIBE openalex.awards.cifar_awards;

In [ ]:
%sql
-- §6.3 Data completeness (runbook canonical query)
-- NOTE: pct_amount expected 0 (CIFAR amount waiver — see header).
-- pct_description expected 100% (built from role_name + program_name).
-- pct_dates expected ~95-100% (uses WP post first_seen_date as proxy).
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(start_date) AS has_start_date,
    COUNT(lead_investigator) AS has_pi,
    ROUND(COUNT(display_name) * 100.0 / COUNT(*), 1) AS pct_title,
    ROUND(COUNT(start_date) * 100.0 / COUNT(*), 1) AS pct_dates,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
    ROUND(COUNT(description) * 100.0 / COUNT(*), 1) AS pct_description
FROM openalex.awards.cifar_awards;

In [ ]:
%sql
-- §6.7 amount/currency — WAIVED for CIFAR. Sanity-check the waiver: rows should be NULL on both.
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies
FROM openalex.awards.cifar_awards;

In [ ]:
%sql
-- Sample inspection
SELECT id, SUBSTRING(display_name, 1, 70) AS title,
       funder_scheme, funding_type, start_year,
       lead_investigator.given_name AS given,
       lead_investigator.family_name AS family,
       SUBSTRING(lead_investigator.affiliation.name, 1, 40) AS affiliation,
       lead_investigator.affiliation.country AS country
FROM openalex.awards.cifar_awards
ORDER BY start_year DESC, family
LIMIT 12;

In [ ]:
%sql
-- Program distribution — expect 25-30 distinct programs with rows
SELECT funder_scheme, COUNT(*) AS rows
FROM openalex.awards.cifar_awards
GROUP BY funder_scheme
ORDER BY rows DESC;

In [ ]:
%sql
-- funding_type split — expect most rows 'research' (Fellow/Chair),
-- minority 'other' (Advisor/Advisory Committee)
SELECT funding_type, COUNT(*) AS rows
FROM openalex.awards.cifar_awards
GROUP BY funding_type
ORDER BY rows DESC;

In [ ]:
%sql
-- Funder struct populated correctly?
SELECT funder.id, funder.display_name, funder.ror_id, funder.doi, COUNT(*) AS rows
FROM openalex.awards.cifar_awards
GROUP BY funder.id, funder.display_name, funder.ror_id, funder.doi;